# Besoin Client 4 – Prédiction de la puissance nominale

Objectif : prédire la colonne `puissance_nominale` (régression) à partir des caractéristiques d'une borne.

Responsable : **Personne 2**

## 1. Imports et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('IRVE_cleaned.csv', low_memory=False)
print(df.shape)
df.head()

## 2. Préparation des données

**Justification des colonnes choisies :** (à remplir)

In [ ]:
TARGET = 'puissance_nominale'

# Features – à adapter selon votre analyse
FEATURES_NUM = ['nbre_pdc', 'consolidated_latitude', 'consolidated_longitude']
FEATURES_CAT = ['implantation_station']  # encoder si utilisé

df_model = df[FEATURES_NUM + FEATURES_CAT + [TARGET]].dropna()

# Encodage des catégorielles
le = LabelEncoder()
df_model['implantation_enc'] = le.fit_transform(df_model['implantation_station'])
joblib.dump(le, 'label_encoder_regress.pkl')

FEATURES_FINAL = FEATURES_NUM + ['implantation_enc']
X = df_model[FEATURES_FINAL]
y = df_model[TARGET]

print(f'Shape X : {X.shape}')
print(f'Distribution cible :\n{y.describe()}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train : {X_train.shape} | Test : {X_test.shape}')

## 3. Entraînement + GridSearchCV

**Justification du choix du modèle :** (à remplir – ex. Random Forest Regressor)

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', RandomForestRegressor(random_state=42))
])

param_grid = {
    'reg__n_estimators': [100, 200],
    'reg__max_depth':    [None, 10, 20],
    'reg__min_samples_split': [2, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print(f'Meilleurs paramètres : {grid.best_params_}')
print(f'Meilleur R² (CV)     : {grid.best_score_:.4f}')

## 4. Métriques d'évaluation

**Justification des métriques :** MAE (erreur absolue, interprétable en kW), RMSE (pénalise les grandes erreurs), R² (part de variance expliquée).

In [ ]:
best = grid.best_estimator_
y_pred = best.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'MAE  : {mae:.2f} kW')
print(f'RMSE : {rmse:.2f} kW')
print(f'R²   : {r2:.4f}')

# Graphique réel vs prédit
plt.figure(figsize=(7,5))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
m = max(y_test.max(), y_pred.max())
plt.plot([0, m], [0, m], 'r--', label='Parfait')
plt.xlabel('Valeur réelle (kW)'); plt.ylabel('Valeur prédite (kW)')
plt.title('Réel vs Prédit – Puissance nominale')
plt.legend(); plt.tight_layout(); plt.show()

## 5. Sauvegarde du modèle

In [ ]:
joblib.dump(best, 'model_regression.pkl')
print('Modèle sauvegardé : model_regression.pkl')

## 6. Conclusion

- Variables choisies + justification : **à rédiger**
- Modèle + principe de fonctionnement : **à rédiger**
- Analyse des métriques : **à rédiger**